# 03 — Trasformare un dataset QA in dati per decision model
Notebook centrale. Usiamo due dataset **pubblici e gratuiti**:
- **PubMedQA** — domande biomediche yes/no/maybe → Noul + Choice + Score.
- **Banking77** — classificazione di intenti a 77 classi → Choice con hard negative + Noul.

L'output è JSONL nello schema documentato in `data/SCHEMA.md`.


In [ ]:
from decision_qa import load_examples

pubmed = load_examples('pubmedqa', split='train', limit=8)
banking = load_examples('banking77', split='train', limit=8)
print('PubMedQA:', len(pubmed), '| Banking77:', len(banking))
print(pubmed[0].question, '->', pubmed[0].label)
print(banking[0].context, '->', banking[0].label)


## Conversione
Per PubMedQA il `final_decision` diventa una Choice a 3 vie, il `long_answer` un enunciato Noul (affiancato a uno contradetto) e una Score a 3 livelli.
Per Banking77 l'intento corretto è mescolato con **hard negative** lessicali e le opzioni sono mescolate.


In [ ]:
from decision_qa import ConversionConfig, convert_pubmedqa, convert_banking77

cfg = ConversionConfig(n_distractors=3, seed=42)
pm_records = convert_pubmedqa(pubmed, cfg, split='train')
bk_records = convert_banking77(banking, cfg, split='train')
print(pm_records[0].model_dump_json(indent=2)[:900])


## Scrittura e validazione
`build_dataset` carica, converte, divide e scrive il JSONL. Verifichiamo le invarianti: indici gold validi, distractor mai uguali all'etichetta corretta, coppie Noul con entrambe le polarità.


In [ ]:
from decision_qa import build_dataset, read_jsonl

path = build_dataset('pubmedqa', 'data/pubmedqa_decisions.jsonl',
                     limits={'train': 40, 'val': 8, 'test': 8}, cfg=cfg)
records = read_jsonl(path)
print('records:', len(records), '| path:', path)

from decision_qa.questions import ChoiceQuestion, NoulQuestion
for r in records:
    for q in r.questions:
        if isinstance(q, ChoiceQuestion):
            assert 0 <= q.gold_index < len(q.options)
        if isinstance(q, NoulQuestion):
            assert q.statement  # both polarities are emitted per record
print('schema invariants OK')


Prossimi passi: addestrare su questi dati (04), valutarli (05). Dettagli: `data/SCHEMA.md`.
